# Approach 7 — text-only retrieval-augmented research context

This notebook retrieves similar reports from the **training split only** and builds a traceable context bundle for review. It does not inspect images, scrape third-party websites, diagnose disease, or claim that retrieved reports are evidence for the current image. Retrieved examples are identified by study UID.

## Run safely

Use a generated draft as the query when available, or enter a plain-text query below. The retrieval index is fit only on training reports. Validation/test reports are never indexed. The response is a list of retrieved examples, not a generated clinical answer.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "config.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the cxr_second_reader project folder in VS Code/Jupyter.")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Local project folder:", REPO_ROOT)


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display
from src.experiment_utils import study_rows

train_df = study_rows("train", None)
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=50000, stop_words="english")
corpus = train_df["target_text"].fillna("").astype(str).tolist()
train_matrix = vectorizer.fit_transform(corpus)
print("Training reports indexed:", len(corpus), "terms:", len(vectorizer.vocabulary_))

RESULTS_DIR = Path(os.environ.get("QWEN_SAVE_DIR", REPO_ROOT / "qwen3vl_cxr")).expanduser()
EVAL_NAME = next((name for name in ["eval_qwen_lora_test_final.csv", "eval_qwen_lora_finetuned.csv", "eval_qwen_lora_validation_current.csv"] if (RESULTS_DIR / name).is_file()), None)
EVAL_PATH = RESULTS_DIR / EVAL_NAME if EVAL_NAME else None
if EVAL_PATH:
    eval_df = pd.read_csv(EVAL_PATH)
    QUERY = str(eval_df.iloc[0]["prediction"])
    print("Using a saved Qwen report draft as query for research review.")
else:
    QUERY = "Findings: The heart size is normal. The lungs are clear. Impression: No acute process."
    print("Using the editable example query below.")

In [ ]:
def retrieve_training_reports(query, top_k=3):
    query_vector = vectorizer.transform([str(query)])
    scores = cosine_similarity(query_vector, train_matrix).ravel()
    top = scores.argsort()[::-1][:top_k]
    return pd.DataFrame({
        "rank": range(1, len(top)+1),
        "uid": train_df.iloc[top]["uid"].astype(str).to_numpy(),
        "similarity": scores[top],
        "indication": train_df.iloc[top]["indication"].astype(str).to_numpy(),
        "retrieved_training_report": train_df.iloc[top]["target_text"].astype(str).to_numpy(),
    })

retrieved = retrieve_training_reports(QUERY, top_k=3)
display(retrieved)

def evidence_bundle(query, examples):
    blocks = ["Research-only retrieved examples; not diagnostic evidence. Verify the source image independently.",
              f"Query draft: {query}"]
    for row in examples.itertuples(index=False):
        blocks.append(f"[Training study UID {row.uid}; similarity {row.similarity:.3f}] {row.retrieved_training_report}")
    return "\n\n".join(blocks)

context = evidence_bundle(QUERY, retrieved)
print(context[:6000])

## Limitations and next step

TF-IDF is lexical retrieval: similar wording does not imply similar pathology or clinical relevance. The retrieved reports are examples from the training corpus, not medical references. A future knowledge RAG should ingest a curated, license-appropriate set of authoritative sources, preserve source URLs and retrieval dates, and be evaluated for unsupported claims before any assistant-style generation is considered.